# (Entry Level) Project_1_Oral_Pill_Recognition - Team 3 (Pill-터링!)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import unicodedata  # 0번 섹션에 추가 필요

# 📌 경로 설정 (제공해주신 경로 반영)
def normalize_path(path):
    # 1. unicodedata.normalize('NFC', path): 경로 문자열을 NFC 방식으로 통일
    # 2. .strip(): 앞뒤에 붙은 불필요한 공백 제거
    return unicodedata.normalize('NFC', path).strip()

In [8]:
############################################################
# 0. 라이브러리 임포트 & 경로 설정
############################################################
import os
import json
import pandas as pd
from PIL import Image
import unicodedata
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torchvision
from torchvision.transforms import v2
from torchvision import tv_tensors
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import wandb

TRAIN_JSON_PATH = os.path.join(extract_path, "merged_annotations_train_final.json")
TEST_JSON_PATH = os.path.join(extract_path, "merged_annotations_test_final.json")
TRAIN_IMG_DIR = os.path.join(extract_path, "train_images")
TEST_IMG_DIR  = os.path.join(extract_path, "test_images")

# merged_annotation json 경로
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

############################################################
# 1. 병합된 JSON 파일을 읽어서 DataFrame으로 만들기
############################################################

def build_df_from_merged_json(json_path, img_dir):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # 1) 이미지 정보 매핑 (id -> file_name)
    id_to_fname = {img["id"]: img["file_name"] for img in data["images"]}

    records = []
    # 2) 어노테이션 순회
    for ann in data["annotations"]:
        img_id_coco = ann["image_id"]
        file_name = id_to_fname.get(img_id_coco)
        
        if file_name is None: continue
        
        img_path = os.path.join(img_dir, file_name)
        
        # 실제 이미지 파일이 있는지 확인 (선택 사항이지만 안전함)
        if not os.path.exists(img_path):
            continue

        x, y, w, h = ann["bbox"]
        
        records.append({
            "image_path": img_path,
            "image_id": os.path.splitext(file_name)[0], # 파일명을 ID로 사용
            "category_id": int(ann["category_id"]),
            "bbox_x": float(x),
            "bbox_y": float(y),
            "bbox_w": float(w),
            "bbox_h": float(h),
        })

    return pd.DataFrame(records)

# 실행
df = build_df_from_merged_json(TRAIN_JSON_PATH, TRAIN_IMG_DIR)
print(f"✅ 학습 데이터 로드 완료: {len(df)} 개의 객체 탐지됨")

✅ 학습 데이터 로드 완료: 4526 개의 객체 탐지됨


In [9]:
############################################################
# 2. category_id 매핑 (겉으로는 안 바꾸고, 모델 내부에서만 사용)
############################################################

# 원본 category_id 집합
unique_cats = sorted(df["category_id"].unique())
print("고유 category_id 개수:", len(unique_cats))

# 내부용: 모델에 넣을 label (1 ~ num_classes-1), 0은 background
# orig2model = {cid: i + 1 for i, cid in enumerate(unique_cats)}   # 원본 → 모델용
orig2model = {cid: i for i, cid in enumerate(unique_cats)}
model2orig = {v: k for k, v in orig2model.items()}               # 모델용 → 원본

num_classes = len(unique_cats) + 1  # background 포함
print("num_classes (background 포함):", num_classes)

고유 category_id 개수: 73
num_classes (background 포함): 74


In [10]:
# wandb 설치 및 로그인
!pip install wandb
import wandb
import os

wandb.login(key=())

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: codeit-project-team3 (codeit-project-team3-pilltering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
############################################################
# wandb 실험 초기화
# ※ run_name을 바꿔가며 실험을 구분하세요 (예: "exp01-baseline", "exp02-augment")
############################################################

config = {
    "run_name"       : "exp1-yolov8s",   # ← 실험마다 변경
    "model"          : "yolov8s",
    "num_epochs"     : 20,  #epoch 변경시
    "batch_size"     : 16, # batch 변경시
    "lr"             : 1e-4, # lr 변경시
    "weight_decay"   : 1e-4, 
    "lr_step_size"   : 3,
    "lr_gamma"       : 0.1,
    "score_threshold": 0.3,
    "augmentation"   : "HFlip+VFlip+ColorJitter+Grayscale+Normalize",
    "num_classes"    : num_classes,
}

wandb.init(
    entity="codeit-project-team3-pilltering",
    project="Pill-터링Project_초급프로젝트",
    name=config["run_name"],
    config=config,
)

print(f"✅ wandb 초기화 완료 | run: {wandb.run.name}")

✅ wandb 초기화 완료 | run: exp04-yolov8s-yj


In [ ]:
# import cv2
# import numpy as np
# import albumentations as A
# from albumentations.pytorch import ToTensorV2
# from torch.utils.data import WeightedRandomSampler

# # ── transforms ──────────────────────────────────────────
# train_transforms = A.Compose([A.Resize(1280, 1280),
#                               A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, border_mode=0, p=0.8),
#                               A.HorizontalFlip(p=0.5),
#                               A.VerticalFlip(p=0.5),
#                               A.RandomRotate90(p=0.5),
#                               A.ColorJitter(brightness=0.2, contrast=0.2,saturation=0.2, hue=0.1, p=0.5),
#                               A.CLAHE(clip_limit=4.0, p=0.4),
#                               A.GaussNoise(p=0.3),
#                               A.MotionBlur(blur_limit=3, p=0.2),
#                               A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#                               ToTensorV2()], 
#                               bbox_params=A.BboxParams(format = 'coco',
#                                                       label_fields   = ['category_ids'],
#                                                       min_area = 1.0,
#                                                       min_visibility = 0.1,
#                                                       clip= True)) # bbox 크기

# val_transforms = A.Compose([A.Resize(1280, 1280),
#                             A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
#                             ToTensorV2()], 
#                             bbox_params=A.BboxParams(format= 'coco',
#                                                       label_fields = ['category_ids'],
#                                                       clip = True))


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [13]:
# ── Dataset ──────────────────────────────────────────────
class OralDrugDataset(Dataset):
    def __init__(self, df, orig2model, transforms=None):
        self.df         = df.reset_index(drop=True)
        self.orig2model = orig2model
        self.transforms = transforms
        self.grouped    = {img_id: grp for img_id, grp in self.df.groupby("image_id")}
        self.image_ids  = list(self.grouped.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        df_img   = self.grouped[image_id]
        img_path = df_img["image_path"].iloc[0]

        # numpy (-> albumentations 필수)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        bboxes, category_ids = [], []
        for _, row in df_img.iterrows():
            w, h = row["bbox_w"], row["bbox_h"]
            if w <= 1 or h <= 1:
                continue
            # ✅ COCO format [x, y, w, h]
            bboxes.append([row["bbox_x"], row["bbox_y"], w, h])
            category_ids.append(self.orig2model[int(row["category_id"])])

        if self.transforms is not None:
            transformed  = self.transforms(image= image, bboxes= bboxes, category_ids = category_ids)
            # ✅ 각각 분리해서 꺼내기
            image        = transformed["image"]
            bboxes       = transformed["bboxes"]
            category_ids = transformed["category_ids"]
        else:
            image = torch.as_tensor(
                image / 255.0, dtype=torch.float32).permute(2, 0, 1)

        # ✅ XYXY 변환 + bboxes/labels 동시 필터링
        boxes_xyxy, valid_labels = [], []
        for (x, y, w, h), label in zip(bboxes, category_ids):
            if (x + w) > x and (y + h) > y:
                boxes_xyxy.append([x, y, x+w, y+h])
                valid_labels.append(label)

        boxes_tensor = (torch.tensor(boxes_xyxy, dtype=torch.float32)
                        if boxes_xyxy
                        else torch.zeros((0, 4), dtype=torch.float32))

        target = {"boxes"   : boxes_tensor,
                  "labels"  : torch.tensor(valid_labels, dtype=torch.int64),
                  "image_id": torch.tensor([idx])}

        return image, target


In [14]:
# ############################################################
# # 3. Dataset 정의
# ############################################################

# class OralDrugDataset(Dataset):
#     """
#     Faster R-CNN용 Dataset
#     - __getitem__은 (image, target) 반환
#     - image: [C,H,W] float tensor
#     - target: dict(boxes, labels, image_id, ...)
#     """
#     def __init__(self, df, orig2model, transforms=None):
#         self.df = df.reset_index(drop=True)
#         self.orig2model = orig2model
#         self.transforms = transforms

#         # 이미지 단위로 그룹을 만들기 위해 고유 image_id 리스트를 유지
#         self.image_ids = self.df["image_id"].unique().tolist()

#     def __len__(self):
#         return len(self.image_ids)

#     def __getitem__(self, idx):
#         # 1) image_id 선택
#         image_id = self.image_ids[idx]

#         # 2) 해당 image_id에 해당하는 모든 row (여러 객체) 가져오기
#         df_img = self.df[self.df["image_id"] == image_id]

#         # 3) 이미지 로드
#         img_path = df_img["image_path"].iloc[0]
#         image = Image.open(img_path).convert("RGB")

#         # 4) bbox / labels 구성
#         boxes = []
#         labels = []

#         for _, row in df_img.iterrows():
#             x = row["bbox_x"]
#             y = row["bbox_y"]
#             w = row["bbox_w"]
#             h = row["bbox_h"]

#             # [x1, y1, x2, y2] 형식으로 변환
#             x1 = x
#             y1 = y
#             x2 = x + w
#             y2 = y + h
#             boxes.append([x1, y1, x2, y2])

#             # 원본 category_id → 모델용 label로 변환
#             orig_cat = int(row["category_id"])
#             model_cat = self.orig2model[orig_cat]
#             labels.append(model_cat)

#         boxes = torch.tensor(boxes, dtype=torch.float32)
#         labels = torch.tensor(labels, dtype=torch.int64)

#         target = {
#             "boxes": boxes,
#             "labels": labels,
#             # image_id는 loss에는 크게 영향 없음, 그냥 인덱스 정도로 넣어도 무방
#             "image_id": torch.tensor([idx]),
#         }

#         if self.transforms is not None:
#             image = self.transforms(image)

#         return image, target


# ############################################################
# # 4. Transform, Dataset, DataLoader 구성
# ############################################################

# train_transforms = T.Compose([
#     # 필요하다면 여기서 RandomHorizontalFlip 등 augmentation 추가
#     T.ToTensor(),  # PIL 이미지를 [0,1] float tensor로 변환
# ])

# dataset = OralDrugDataset(df, orig2model=orig2model, transforms=train_transforms)

# # 간단히 train/val split (예: 90% train, 10% val)
# indices = torch.randperm(len(dataset)).tolist()
# split = int(0.9 * len(indices))
# train_indices = indices[:split]
# val_indices   = indices[split:]

# train_dataset = torch.utils.data.Subset(dataset, train_indices)
# val_dataset   = torch.utils.data.Subset(dataset, val_indices)

# def collate_fn(batch):
#     # detection 모델용 collate_fn: 리스트의 튜플을 튜플의 리스트로
#     return tuple(zip(*batch))

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=2,              # GPU 메모리에 맞게 조정
#     shuffle=True,
#     collate_fn=collate_fn,
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=2,
#     shuffle=False,
#     collate_fn=collate_fn,
# )

# print("train steps:", len(train_loader), "val steps:", len(val_loader))



In [15]:

# ############################################################
# # 7. test_images에 대해 예측 → submission.csv 생성
# ############################################################

# # test 이미지 파일 목록 가져오기
# test_files = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")]
# test_files = sorted(test_files)

# model.eval()

# rows = []
# annotation_id = 1      # submission용 annotation_id 시작
# score_threshold = 0.3  # 너무 낮은 점수는 제거 (필요에 따라 조정)

# with torch.no_grad():
#     for f in test_files:
#         img_path = os.path.join(TEST_IMG_DIR, f)
#         image = Image.open(img_path).convert("RGB")

#         # image_id = 파일명에서 확장자 제거한 문자열 그대로 사용
#         image_id = os.path.splitext(f)[0]

#         img_tensor = T.ToTensor()(image).to(DEVICE)
#         outputs = model([img_tensor])[0]

#         keep = outputs["scores"].cpu() >= score_threshold
#         boxes  = outputs["boxes"].cpu()[keep]
#         labels = outputs["labels"].cpu()[keep]
#         scores = outputs["scores"].cpu()[keep]

#         for box, lab, sc in zip(boxes, labels, scores):
#             x1, y1, x2, y2 = box.tolist()
#             w = x2 - x1
#             h = y2 - y1

#             orig_cat = model2orig[int(lab.item())]

#             rows.append({
#                 "annotation_id": annotation_id,
#                 "image_id": image_id,  # 문자열 그대로 사용
#                 "category_id": orig_cat,
#                 "bbox_x": x1,
#                 "bbox_y": y1,
#                 "bbox_w": w,
#                 "bbox_h": h,
#                 "score": float(sc.item()),
#             })

#             annotation_id += 1

# # DataFrame으로 만들고 저장
# df_sub = pd.DataFrame(rows, columns=[
#     "image_id", "category_id",
#     "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"
# ])
# # 이미지 ID별로 점수 높은 순 정렬 후 상위 4개 추출
# df_sub = df_sub.sort_values(by=["image_id", "score"], ascending=[True, False])
# df_sub = df_sub.groupby("image_id").head(4)

# # 3) 최종 annotation_id 부여 (1부터 순차적으로)
# df_sub.insert(0, "annotation_id", range(1, len(df_sub) + 1))

# # 4) CSV 저장
# output_path = "final_submission.csv"
# df_sub.to_csv(os.path.join(extract_path, output_path), index=False)

# print(f"✅ 생성 완료: {output_path}")
# print(f"📊 총 예측 객체 수: {len(df_sub)}")
# print(df_sub.head())

In [16]:
# ############################################################
# # 8. 모델 성능 평가 (mAP 측정)
# ############################################################

# import json
# from pycocotools.coco import COCO
# from pycocotools.cocoeval import COCOeval

# # 1. df_sub(Pandas)를 COCO 평가용 리스트로 변환
# eval_results = []
# for _, row in df_sub.iterrows():
#     eval_results.append({
#         "image_id": int(row["image_id"]),
#         "category_id": int(row["category_id"]),
#         "bbox": [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
#         "score": float(row["score"])
#     })

# # 2. 임시 JSON 파일로 저장
# temp_json = "temp_eval.json"
# with open(temp_json, "w") as f:
#     json.dump(eval_results, f)

# # 3. COCO 평가 실행
# coco_gt = COCO(TEST_JSON_PATH)
# coco_dt = coco_gt.loadRes(temp_json)

# coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
# coco_eval.evaluate()
# coco_eval.accumulate()
# coco_eval.summarize()

---
## [실험] YOLO v8 / v11 백본 교체 실험
> **Faster R-CNN 셀을 실행하지 않고 이 셀들만 독립 실행 가능**
> (단, **0번 셀(import·경로 설정)**, **bbox 필터링(df)**, **category 매핑(orig2model)** 셀은 먼저 실행 필요)
---

In [17]:
# ultralytics 설치
!pip install ultralytics -q
from ultralytics import YOLO
import ultralytics
print("ultralytics 버전:", ultralytics.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.0 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics 버전: 8.4.27


In [19]:
############################################################
# [YOLO] 데이터셋 준비: COCO → YOLO 형식 변환
############################################################
import os, shutil, random
from PIL import Image

random.seed(42)

YOLO_BASE = os.path.join(extract_path, "yolo_dataset")
for split in ["train", "val"]:
    os.makedirs(os.path.join(YOLO_BASE, "images", split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_BASE, "labels", split), exist_ok=True)

image_ids = df["image_id"].unique().tolist()
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.9)
train_ids = set(image_ids[:split_idx])
val_ids   = set(image_ids[split_idx:])
print(f"train: {len(train_ids)}장 | val: {len(val_ids)}장")

for img_id in image_ids:
    df_img   = df[df["image_id"] == img_id]
    img_path = df_img["image_path"].iloc[0]
    split    = "train" if img_id in train_ids else "val"

    fname   = os.path.basename(img_path)
    dst_img = os.path.join(YOLO_BASE, "images", split, fname)
    if not os.path.exists(dst_img):
        shutil.copy(img_path, dst_img)

    with Image.open(img_path) as img:
        W, H = img.size

    txt_name = os.path.splitext(fname)[0] + ".txt"
    dst_lbl  = os.path.join(YOLO_BASE, "labels", split, txt_name)

    with open(dst_lbl, "w") as lf:
        for _, row in df_img.iterrows():
            orig_cat = int(row["category_id"])
            bw, bh = row["bbox_w"], row["bbox_h"]
            if bw <= 1 or bh <= 1:
                continue
            yolo_cls = orig2model[orig_cat]  # ✅ 0-based
            cx = (row["bbox_x"] + bw / 2) / W
            cy = (row["bbox_y"] + bh / 2) / H
            nw = bw / W
            nh = bh / H
            lf.write(f"{yolo_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\n")

# data.yaml 생성
yaml_path = os.path.join(YOLO_BASE, "data.yaml")
with open(yaml_path, "w") as yf:
    yf.write(f"path: {YOLO_BASE}\n")
    yf.write("train: images/train\n")
    yf.write("val: images/val\n")
    yf.write(f"nc: {len(unique_cats)}\n")
    yf.write(f"names: {list(range(len(unique_cats)))}\n")

print("data.yaml 생성 완료:", yaml_path)


train: 1340장 | val: 149장
data.yaml 생성 완료: /content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_dataset/data.yaml


In [ ]:
############################################################
# [YOLO] 학습
# 모델 선택: yolov8n(빠름) / yolov8s(균형) / yolo11n / yolo11s
# GPU 메모리 부족 시 batch를 4로 줄이기
############################################################
from ultralytics import YOLO

yolo_model = YOLO("yolov8s.pt")  # ← 모델명만 바꿔서 실험 가능

yaml_path = os.path.join(extract_path, "yolo_dataset", "data.yaml")

results = yolo_model.train(
    data     = yaml_path,
    epochs   = 20,
    imgsz    = 1280,
    batch    = 16,
    device   = 0,
    project  = extract_path,
    name     = "yolo_oral_drug",
    exist_ok = True,
    patience = 10,
    save     = True,
    plots    = True,
)

best_pt = os.path.join(extract_path, "yolo_oral_drug", "weights", "best.pt")
print("학습 완료. best model:", best_pt)

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/data/초급_프로젝트/dataset/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_oral_drug, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, 

In [21]:
############################################################
# [YOLO] 추론 → submission.csv 생성
# ✅ 베이스라인과 동일: TEST_IMG_DIR 사용, 포맷 동일
############################################################
from ultralytics import YOLO
import pandas as pd

best_model   = YOLO(best_pt)
test_files   = sorted([f for f in os.listdir(TEST_IMG_DIR) if f.endswith(".png")])
SCORE_THRESH = 0.3  # 낮으면 오탐 증가 / 높으면 미탐 증가

rows_yolo     = []
annotation_id = 1

for fname in test_files:
    img_path = os.path.join(TEST_IMG_DIR, fname)
    image_id = os.path.splitext(fname)[0]  # ✅ 베이스라인과 동일한 image_id 형식

    preds = best_model.predict(img_path, conf=SCORE_THRESH, verbose=False)

    for r in preds:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls[0].item())    # YOLO 0-based class
            score  = float(box.conf[0].item())

            orig_cat = model2orig[cls_id]

            rows_yolo.append({
                "annotation_id": annotation_id,
                "image_id"     : image_id,
                "category_id"  : orig_cat + 1,  # ✅ 베이스라인 포맷과 동일
                "bbox_x"       : x1,
                "bbox_y"       : y1,
                "bbox_w"       : x2 - x1,
                "bbox_h"       : y2 - y1,
                "score"        : score,
            })
            annotation_id += 1

df_sub_yolo = (
    pd.DataFrame(rows_yolo)
    .sort_values(["image_id", "score"], ascending=[True, False])
    .groupby("image_id").head(4)
    .reset_index(drop=True)
)
df_sub_yolo = df_sub_yolo.drop(columns=["annotation_id"])
df_sub_yolo.insert(0, "annotation_id", range(1, len(df_sub_yolo) + 1))

output_yolo = "baseline_submission_yolo.csv"
df_sub_yolo.to_csv(os.path.join(extract_path, output_yolo), index=False)
print(f"생성 완료: {output_yolo}")
print(f"총 예측 객체 수: {len(df_sub_yolo)}")
print(df_sub_yolo.head())

생성 완료: baseline_submission_yolo.csv
총 예측 객체 수: 3220
   annotation_id image_id  category_id      bbox_x      bbox_y      bbox_w  \
0              1        1         1900  158.252945  251.284088  202.356796   
1              2        1        27926  600.142212  674.883545  254.770996   
2              3        1        16551  557.450073   71.632324  396.739502   
3              4        1        24850  173.555786  741.861816  178.046509   
4              5       10        21771  643.630371  289.973022  187.957764   

       bbox_h     score  
0  125.698914  0.854414  
1  478.216309  0.832301  
2  403.677307  0.799911  
3  291.604492  0.490261  
4  182.199097  0.967309  


In [22]:
############################################################
# [YOLO] mAP 평가 (pycocotools)
############################################################
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

eval_results_yolo = [
    {
        "image_id"   : int(row["image_id"]),
        "category_id": int(row["category_id"]-1),  # ✅ 모델 내부에서 0-based → 원본 category_id로 변환
        "bbox"       : [row["bbox_x"], row["bbox_y"], row["bbox_w"], row["bbox_h"]],
        "score"      : float(row["score"])
    }
    for _, row in df_sub_yolo.iterrows()
]

temp_json_yolo = "temp_eval_yolo.json"
with open(temp_json_yolo, "w") as f:
    json.dump(eval_results_yolo, f)

coco_gt   = COCO(TEST_JSON_PATH)
coco_dt   = coco_gt.loadRes(temp_json_yolo)
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

stats = coco_eval.stats
print("\n===== 성능 =====")
print(f"YOLO (yolov8s)                     mAP@0.5:0.95 = {stats[0]:.3f}  mAP@0.5 = {stats[1]:.3f}")

loading annotations into memory...
Done (t=0.32s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.01s).
Accumulating evaluation results...
DONE (t=0.48s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.372
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.378
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.377
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.372
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.969
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.969
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe